# Can we predict late deliveries?
Target: `late_delivery`. Prediction is made at dispatch, for future consignments.

Design decisions (per checklist.md):
- **Features known at dispatch only**: `carrier`, `distance_km`, `weight_kg`, `parcels`, plus calendar features from `dispatched_on`. `claim_paid_eur` is recorded after delivery (a claim settled for the consignment) and is a consequence of lateness, so it is **excluded** (leakage). `consignment` is an identifier and is dropped.
- **Split**: each consignment appears once, so no entity grouping is needed; the model is meant for later rows, so the test set is the **latest 20%** by `dispatched_on`.
- **CV**: expanding-window `TimeSeriesSplit` inside the training rows only; preprocessing lives in a Pipeline fitted per fold.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
from sklearn.base import clone

df = pd.read_csv('data.csv', parse_dates=['dispatched_on']).sort_values('dispatched_on', kind='stable').reset_index(drop=True)
assert df.consignment.is_unique
print(df.shape, df.dispatched_on.min(), df.dispatched_on.max(), df.late_delivery.mean())

## 1. Hold out the test set (latest 20% by time)

In [ ]:
cut = int(len(df)*0.8)
train, test = df.iloc[:cut].copy(), df.iloc[cut:].copy()
print(train.dispatched_on.max(), '<=', test.dispatched_on.min())
print('train late rate', train.late_delivery.mean(), '| test rows', len(test))
# Exploration below uses the training rows only; the test set is not touched until step 5.
print(train.groupby('carrier').late_delivery.agg(['mean','size']))
print(train.groupby(pd.qcut(train.distance_km,5)).late_delivery.mean())

## 2. Features and candidate models (declared before any scoring)
Features: `carrier` (one-hot), `distance_km`, `weight_kg`, `parcels`, `hour`, `dayofweek` (derived from dispatch time).

Candidates:
1. **baseline** - predicts the training prior (trivial).
2. **logreg** - logistic regression, standardised numerics, small grid over `C` in {0.1, 1, 10}.
3. **hgb** - HistGradientBoosting, small grid over `max_depth` in {2, 3} and `learning_rate` in {0.05, 0.1}.

Selection metric: mean ROC AUC over time-ordered CV folds. Decision threshold: chosen on pooled out-of-fold predictions to maximise F1.

In [ ]:
def add_time(X):
    X = X.copy()
    X['hour'] = X.dispatched_on.dt.hour
    X['dayofweek'] = X.dispatched_on.dt.dayofweek
    return X.drop(columns='dispatched_on')
NUM = ['distance_km','weight_kg','parcels','hour','dayofweek']; CAT = ['carrier']
FEATS = ['dispatched_on','carrier','distance_km','weight_kg','parcels']

def make(model, scale):
    pre = ColumnTransformer([('num', StandardScaler() if scale else 'passthrough', NUM),
                             ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])
    return Pipeline([('t', FunctionTransformer(add_time)), ('pre', pre), ('m', model)])

candidates = {'baseline': [make(DummyClassifier(strategy='prior'), False)]}
candidates['logreg'] = [make(LogisticRegression(C=c, max_iter=1000), True) for c in (0.1,1,10)]
candidates['hgb'] = [make(HistGradientBoostingClassifier(max_depth=d, learning_rate=lr, random_state=0), False)
                     for d in (2,3) for lr in (0.05,0.1)]
Xtr, ytr = train[FEATS].reset_index(drop=True), train.late_delivery.reset_index(drop=True)

## 3. Time-ordered cross-validation on training rows

In [ ]:
cv = TimeSeriesSplit(n_splits=5)
def run_cv(p):
    aucs, oof_idx, oof_p = [], [], []
    for a, b in cv.split(Xtr):
        m = clone(p).fit(Xtr.iloc[a], ytr.iloc[a])   # preprocessing fitted on fold-train only
        pr = m.predict_proba(Xtr.iloc[b])[:,1]
        aucs.append(roc_auc_score(ytr.iloc[b], pr)); oof_idx += list(b); oof_p += list(pr)
    return np.mean(aucs), np.std(aucs), np.array(oof_idx), np.array(oof_p)
rows, store = [], {}
for name, ps in candidates.items():
    for i, p in enumerate(ps):
        mu, sd, idx, pr = run_cv(p)
        params = {k:v for k,v in p.named_steps['m'].get_params().items() if k in ('C','max_depth','learning_rate')}
        rows.append((name, str(params), mu, sd)); store[(name,i)] = (mu, idx, pr)
res = pd.DataFrame(rows, columns=['model','params','cv_auc_mean','cv_auc_sd']).sort_values('cv_auc_mean', ascending=False)
res

## 4. Choose model and threshold on CV only

In [ ]:
best_key = max(store, key=lambda k: store[k][0])
best_name, best_i = best_key
_, idx, pr = store[best_key]
y_oof = ytr.iloc[idx].values
grid = np.linspace(0.05, 0.95, 91)
f1s = [f1_score(y_oof, pr>=t) for t in grid]
THRESH = float(grid[int(np.argmax(f1s))])
CHOSEN = candidates[best_name][best_i]
print('CHOSEN:', best_name, CHOSEN.named_steps['m'].get_params().get('C'), CHOSEN.named_steps['m'].get_params().get('max_depth'),
      '| threshold', THRESH, '| CV F1 at threshold', max(f1s), '| CV AUC', store[best_key][0])
print('baseline CV AUC', store[('baseline',0)][0])

The model, features, hyperparameters and threshold above are now frozen. The test set is scored once below.

## 5. Refit on all training rows; score once on the test set

In [ ]:
final = clone(CHOSEN).fit(Xtr, ytr)
Xte, yte = test[FEATS], test.late_delivery
p = final.predict_proba(Xte)[:,1]
base = clone(candidates['baseline'][0]).fit(Xtr, ytr).predict_proba(Xte)[:,1]
out = pd.DataFrame({'model':[best_name,'baseline (reference)'],
  'ROC AUC':[roc_auc_score(yte,p), roc_auc_score(yte,base)],
  'PR AUC':[average_precision_score(yte,p), average_precision_score(yte,base)],
  'Brier':[brier_score_loss(yte,p), brier_score_loss(yte,base)],
  'F1 @ chosen thr':[f1_score(yte,p>=THRESH), f1_score(yte,base>=THRESH)]})
print('test late rate', yte.mean()); out

## Conclusion
See the test table above for the headline (single scoring of the held-out latest 20%). `claim_paid_eur` was deliberately excluded: it would nearly perfectly separate the classes but is only known after delivery, so it could not be used at dispatch.

## Caveat: the test period looks different (diagnostic after scoring; nothing was re-tuned)
Test headline (hgb, max_depth=2, lr=0.05, threshold 0.23): ROC AUC 0.68, PR AUC 0.051 vs prevalence 0.022, F1 0.065. CV AUC was 0.72, so the ranking signal (distance, carrier) carries over only partly.

The late rate is ~26% in training but only 2.2% in the test period. Monthly rates fall from ~18% (Sep 2025) to ~5% (Oct 2025) and ~1% (Jan-Feb 2026). Either lateness has genuinely collapsed or recent outcomes are not yet fully recorded (right-censoring). The data cannot distinguish these. Under either explanation, the threshold chosen on older data (0.23) is far too low for the test period, which is why F1 is poor. Check how `late_delivery` is populated for recent rows before relying on this model.
